# CalCOFI — Su Sıcaklığı Tahmin Analizi

**Veri seti:** [CalCOFI: Over 60 Years of Oceanographic Data](https://www.kaggle.com/sohier/calcofi)  
**Soru:** Tuzluluk, derinlik ve çözünmüş oksijen verileri kullanılarak su sıcaklığı tahmin edilebilir mi?

---

## İçerik
1. Kütüphaneler ve veri yükleme
2. Keşifsel veri analizi (EDA)
3. Model 1 — Basit lineer regresyon (Salnty)
4. Model 2 — Çok değişkenli regresyon (Salnty + Depthm)
5. Model 3 — Çok değişkenli regresyon (Salnty + Depthm + O2ml_L)
6. Model karşılaştırması
7. Korelasyon ve multicollinearity analizi
8. Sonuçlar

## 1. Kütüphaneler ve Veri Yükleme

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Kütüphaneler yüklendi.')

In [ ]:
# Google Drive'dan yükleme (Colab)
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/bottle.csv', low_memory=False)

print(f'Boyut: {df.shape[0]:,} satır, {df.shape[1]} sütun')
df[['Salnty', 'Depthm', 'O2ml_L', 'T_degC']].head()

## 2. Keşifsel Veri Analizi (EDA)

In [ ]:
# Kullanacağımız dört değişkenin özet istatistikleri
cols = ['Salnty', 'Depthm', 'O2ml_L', 'T_degC']
df[cols].describe().round(2)

In [ ]:
# Eksik değer oranları
missing = df[cols].isnull().mean().mul(100).round(1)
print('Eksik değer oranları (%)')
print(missing.to_string())

In [ ]:
# Tuzluluk ve sıcaklık ilişkisi — ham görünüm
df_clean = df[['Salnty', 'T_degC']].dropna()

plt.scatter(df_clean['Salnty'], df_clean['T_degC'],
            alpha=0.1, s=1, color='steelblue')
plt.xlabel('Tuzluluk (Salnty)')
plt.ylabel('Sıcaklık (T_degC)')
plt.title('Tuzluluk ve Su Sıcaklığı — Ham Dağılım')
plt.tight_layout()
plt.show()

print(f"Korelasyon (Salnty ↔ T_degC): {df_clean['Salnty'].corr(df_clean['T_degC']):.4f}")

## 3. Model 1 — Basit Lineer Regresyon

**Bağımsız değişken:** Salnty (tuzluluk)  
**Hedef değişken:** T_degC (su sıcaklığı)

In [ ]:
df1 = df[['Salnty', 'T_degC']].dropna()

X1 = df1[['Salnty']]
y1 = df1['T_degC']

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42)

model1 = LinearRegression()
model1.fit(X1_train, y1_train)
y1_pred = model1.predict(X1_test)

r2_1  = r2_score(y1_test, y1_pred)
rmse_1 = np.sqrt(mean_squared_error(y1_test, y1_pred))

print('Model 1 — Sadece Tuzluluk')
print(f'  Katsayı (slope)   : {model1.coef_[0]:.4f}')
print(f'  Sabit (intercept) : {model1.intercept_:.4f}')
print(f'  R²                : {r2_1:.4f}')
print(f'  RMSE              : {rmse_1:.4f} °C')
print(f'\nDenklem: T_degC = {model1.intercept_:.2f} + ({model1.coef_[0]:.2f} × Salnty)')

In [ ]:
plt.scatter(X1_test, y1_test, alpha=0.1, s=1,
            color='steelblue', label='Gerçek veri')
plt.plot(X1_test.sort_values('Salnty'),
         model1.predict(X1_test.sort_values('Salnty')),
         color='red', linewidth=2, label='Regresyon doğrusu')
plt.xlabel('Tuzluluk (Salnty)')
plt.ylabel('Sıcaklık (T_degC)')
plt.title(f'Model 1 — Lineer Regresyon  |  R² = {r2_1:.2f}  |  RMSE = {rmse_1:.2f} °C')
plt.legend()
plt.tight_layout()
plt.show()

**Gözlem:** Regresyon doğrusu verinin gerçek aralığı dışına (Salnty < 32.5) taşıyor — **extrapolation** riski var. Veri 'fan' şeklinde dağılmış, tek değişken yetersiz (R²=0.25).

## 4. Model 2 — Tuzluluk + Derinlik

In [ ]:
df2 = df[['Salnty', 'Depthm', 'T_degC']].dropna()

X2 = df2[['Salnty', 'Depthm']]
y2 = df2['T_degC']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42)

model2 = LinearRegression()
model2.fit(X2_train, y2_train)
y2_pred = model2.predict(X2_test)

r2_2   = r2_score(y2_test, y2_pred)
rmse_2 = np.sqrt(mean_squared_error(y2_test, y2_pred))

print('Model 2 — Tuzluluk + Derinlik')
for name, coef in zip(X2.columns, model2.coef_):
    print(f'  {name:10s} katsayısı: {coef:.4f}')
print(f'  R²  : {r2_2:.4f}')
print(f'  RMSE: {rmse_2:.4f} °C')
print(f'\nNot: Her 1000m derinlikte ~{abs(model2.coef_[1])*1000:.1f}°C soğuma bekleniyor.')

## 5. Model 3 — Tuzluluk + Derinlik + Çözünmüş Oksijen

In [ ]:
df3 = df[['Salnty', 'Depthm', 'O2ml_L', 'T_degC']].dropna()

X3 = df3[['Salnty', 'Depthm', 'O2ml_L']]
y3 = df3['T_degC']

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42)

model3 = LinearRegression()
model3.fit(X3_train, y3_train)
y3_pred = model3.predict(X3_test)

r2_3   = r2_score(y3_test, y3_pred)
rmse_3 = np.sqrt(mean_squared_error(y3_test, y3_pred))

print('Model 3 — Tuzluluk + Derinlik + Oksijen')
for name, coef in zip(X3.columns, model3.coef_):
    print(f'  {name:10s} katsayısı: {coef:.4f}')
print(f'  R²  : {r2_3:.4f}')
print(f'  RMSE: {rmse_3:.4f} °C')
print('\n⚠️  Uyarı: Salnty katsayısı pozitife döndü → multicollinearity şüphesi!')

## 6. Model Karşılaştırması

In [ ]:
karsilastirma = pd.DataFrame({
    'Model'      : ['Model 1 (Salnty)', 'Model 2 (+Depthm)', 'Model 3 (+O2ml_L)'],
    'R²'         : [round(r2_1, 4), round(r2_2, 4), round(r2_3, 4)],
    'RMSE (°C)'  : [round(rmse_1, 4), round(rmse_2, 4), round(rmse_3, 4)],
    'Güvenilirlik': ['Düşük', 'Orta ✓', 'Yüksek R² ama dikkat!']
})
print(karsilastirma.to_string(index=False))

# Görsel karşılaştırma
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

models = ['Model 1\n(Salnty)', 'Model 2\n(+Depthm)', 'Model 3\n(+O2ml_L)']
r2s    = [r2_1, r2_2, r2_3]
rmses  = [rmse_1, rmse_2, rmse_3]
colors = ['#aac4e0', '#5a9fd4', '#1a6aaa']

axes[0].bar(models, r2s, color=colors, width=0.5)
axes[0].set_title('R² Skoru (yüksek = iyi)')
axes[0].set_ylim(0, 1)
for i, v in enumerate(r2s):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=11)

axes[1].bar(models, rmses, color=colors, width=0.5)
axes[1].set_title('RMSE — °C cinsinden hata (düşük = iyi)')
for i, v in enumerate(rmses):
    axes[1].text(i, v + 0.03, f'{v:.2f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 7. Korelasyon ve Multicollinearity Analizi

In [ ]:
corr = df3[['Salnty', 'Depthm', 'O2ml_L', 'T_degC']].corr().round(2)
print('Korelasyon Matrisi')
print(corr.to_string())

plt.figure(figsize=(6, 5))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True

sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1,
            linewidths=0.5, square=True)
plt.title('Değişkenler Arası Korelasyon')
plt.tight_layout()
plt.show()

**Multicollinearity tespiti:**

| İlişki | Korelasyon | Yorum |
|---|---|---|
| O2ml_L ↔ T_degC | +0.80 | Çok güçlü — oksijen en iyi öngörücü |
| O2ml_L ↔ Salnty | -0.82 | Çok güçlü — multicollinearity kaynağı |
| Depthm ↔ T_degC | -0.67 | Güçlü — derinlik önemli |
| Salnty ↔ T_degC | -0.51 | Orta |

`O2ml_L` hem `Salnty` hem `T_degC` ile ~0.80+ korelasyona sahip.  
Bu nedenle Model 3'te `Salnty` katsayısı **-4.63'ten +5.11'e** döndü — matematiksel çakışma, gerçek fiziksel etki değil.  

**Sonuç:** Yorumlanabilirlik için Model 2, tahmin gücü için Model 3 tercih edilebilir.

## 8. Sonuçlar

### Ne bulduk?

- Tuzluluk ile su sıcaklığı arasında **negatif bir ilişki** var (r = -0.51). Derin, soğuk okyanuslar daha tuzlu olduğundan bu beklenen bir sonuç.
- Tek değişkenli model (Model 1, R²=0.25) tuzluluğun tek başına yetersiz olduğunu kanıtladı.
- Derinlik eklenince (Model 2, R²=0.48) model belirgin şekilde güçlendi ve **gizli değişken etkisi** ortaya çıktı — Salnty katsayısı -4.63'ten -1.62'ye düştü.
- Çözünmüş oksijen eklenince (Model 3, R²=0.79) en yüksek tahmin gücüne ulaşıldı ancak **multicollinearity** nedeniyle katsayılar güvenilir olmaktan çıktı.

### Öğrenilen kavramlar

| Kavram | Ne anlama geliyor |
|---|---|
| R² skoru | Modelin hedef değişkendeki varyansı ne kadar açıkladığı |
| RMSE | Tahmin hatasının ortalama büyüklüğü (aynı birimde) |
| Gizli değişken | Hem bağımsız hem bağımlı değişkenle ilişkili, modele dahil edilmemiş faktör |
| Multicollinearity | Bağımsız değişkenlerin birbiriyle güçlü ilişkili olması — katsayıları güvenilmez kılar |
| Extrapolation | Modelin veri aralığı dışında tahmin yapması — güvenilmez |

### Sonraki adımlar
- Polinom regresyon ile eğrisel ilişkiyi yakalamak
- VIF (Variance Inflation Factor) ile multicollinearity'yi sayısal ölçmek
- Random Forest ile lineer olmayan ilişkileri modellemek